<a href="https://colab.research.google.com/github/hafswa-sufi/SemProject_MLEngine/blob/main/01_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from google.colab import drive
drive.mount('/content/drive')

import os, re
import pandas as pd

FOLDER = '/content/drive/MyDrive/INTERNSHIP_MATCH PROJECT/AI Datasets/AI Datasets'
OUT_DIR = os.path.join(FOLDER, 'clean')
os.makedirs(OUT_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
PATH = os.path.join(FOLDER, 'Resume.csv')
df = pd.read_csv(PATH)
print(df.shape)

(2484, 4)


In [16]:
display(df.tail(20))
df.info()
print((df.isna().mean()*100).round(2))
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate resume text:", df['Resume_str'].duplicated().sum())
print("Duplicate IDs:", df['ID'].duplicated().sum())
empty = df['Resume_str'].isna() | (df['Resume_str'].str.strip() == '')
print("Empty resumes:", empty.sum())

,ID,Resume_str,Resume_html,Category
2464,77626587,FLIGHT MANAGER Summary I am ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2465,98389424,GUNNERY SERGEANT Core Qua...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2466,38663892,MANAGEMENT AND PROGRAM ANALYSIS ...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2467,17152392,SUPPLY CHAIN MANAGER Summar...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2468,49646155,ENTREPRENEUR GENERAL MANAGER Su...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2469,33627938,PERSONNEL SECURITY SPECIALIST S...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2470,12904972,"MANAGER, CONTINUOUS IMPROVEMENT ...","<div class=""fontsize fontface vmargins hmargin...",AVIATION
2471,12239749,SPECIALIST OF INFANTRY Professi...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2472,16279537,SUPERVISORY LOGISTICS MANAGEMENT SPEC...,"<div class=""fontsize fontface vmargins hmargin...",AVIATION
2473,17008356,ACCOUNT EXECUTIVE Professiona...,"<div class=""skn-cbg1 fontsize fontface vmargin...",AVIATION


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB
ID             0.0
Resume_str     0.0
Resume_html    0.0
Category       0.0
dtype: float64
Duplicate rows: 0
Duplicate resume text: 2
Duplicate IDs: 0
Empty resumes: 1


In [17]:
counts = df['Category'].value_counts()
print("Categories:", df['Category'].nunique())
print(counts)
print("Largest/smallest ratio:", round(counts.max()/counts.min(), 2))

Categories: 24
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64
Largest/smallest ratio: 5.45


In [18]:
import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [19]:
df = pd.read_csv(PATH)
empty = df['Resume_str'].isna() | (df['Resume_str'].str.strip() == '')

n0 = len(df)
df = df.drop(columns=['Resume_html'], errors='ignore')
df = df[~empty].reset_index(drop=True)
df = df.drop_duplicates(subset='Resume_str').reset_index(drop=True)
df['Resume_clean'] = df['Resume_str'].apply(clean_text)
df = df[df['Resume_clean'].str.len() > 0].reset_index(drop=True)
print("Rows before:", n0, "after:", len(df))

i = 0
print("BEFORE:", df.loc[i, 'Resume_str'][:300])
print("AFTER: ", df.loc[i, 'Resume_clean'][:300])

Rows before: 2484 after: 2481
BEFORE:          HR ADMINISTRATOR/MARKETING ASSOCIATE

HR ADMINISTRATOR       Summary     Dedicated Customer Service Manager with 15+ years of experience in Hospitality and Customer Service Management.   Respected builder and leader of customer-focused teams; strives to instill a shared, enthusiastic commit
AFTER:  hr administrator marketing associate hr administrator summary dedicated customer service manager with 15+ years of experience in hospitality and customer service management respected builder and leader of customer focused teams strives to instill a shared enthusiastic commitment to customer service 


In [20]:
SECTION_WORDS = ('summary','skills','skill','experience','education','objective',
                 'profile','highlights','qualifications','career','professional')

def is_title_chunk(chunk):
    letters = [c for c in chunk if c.isalpha()]
    return (letters and len(chunk) <= 80
            and sum(c.isupper() for c in letters)/len(letters) > 0.8
            and not chunk.lower().startswith(SECTION_WORDS))

def split_title(raw, max_chunks=2):
    rest = raw.strip()
    removed = []
    for _ in range(max_chunks):
        parts = re.split(r'\s{2,}', rest, maxsplit=1)
        if len(parts) == 2 and is_title_chunk(parts[0]):
            removed.append(parts[0])
            rest = parts[1]
        else:
            break
    return (' | '.join(removed) if removed else None), rest

res = df['Resume_str'].apply(split_title)
df['has_title'] = res.apply(lambda x: x[0] is not None)
df['Resume_notitle'] = res.apply(lambda x: clean_text(x[1]))

print("Resumes starting with a title:", df['has_title'].sum(), "of", len(df))
print("Empty notitle:", (df['Resume_notitle'].str.len() == 0).sum())

# check a spread across categories, not just HR
sample = df.sample(8, random_state=1)
for _, r in sample.iterrows():
    print(r['Category'], '| REMOVED:', split_title(r['Resume_str'])[0])
    print('   NOTITLE STARTS:', r['Resume_notitle'][:90], '\n')

# the resumes where no title was found
print(df.loc[~df['has_title'], 'Resume_str'].head(8).str[:100].tolist())

Resumes starting with a title: 2438 of 2481
Empty notitle: 0
CONSTRUCTION | REMOVED: SENIOR CONSTRUCTION MANAGER
   NOTITLE STARTS: summary construction manager on site project manager project manager versed in all aspects 

SALES | REMOVED: SALES ASSOCIATE
   NOTITLE STARTS: experience 08 2014 to current sales associate company name city state greet customers and  

DIGITAL-MEDIA | REMOVED: DIRECTOR OF DIGITAL TRANSFORMATION
   NOTITLE STARTS: executive profile digital and print media professional and consultant with a strong vision 

APPAREL | REMOVED: OFFICE MANAGER - 40+ HOURS PER WEEK
   NOTITLE STARTS: professional summary human recourses office manager with 4 years background in employee ma 

FINANCE | REMOVED: FINANCE SPECIALIST
   NOTITLE STARTS: summary strategic finance accounting professional with experience in extensive contracts a 

DESIGNER | REMOVED: SENIOR INTERIOR DESIGNER
   NOTITLE STARTS: highlights microsoft office suite apple iwork suite autodesk autocad architec

In [21]:
out_file = os.path.join(OUT_DIR, 'resume_clean.csv')
df[['ID','Category','Resume_clean','Resume_notitle']].to_csv(out_file, index=False)
print("Saved:", out_file)

Saved: /content/drive/MyDrive/INTERNSHIP_MATCH PROJECT/AI Datasets/AI Datasets/clean/resume_clean.csv


In [22]:
SRC_DIR = os.path.join(FOLDER, 'src')
os.makedirs(SRC_DIR, exist_ok=True)

code = r'''import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text
'''
with open(os.path.join(SRC_DIR, 'clean_text.py'), 'w') as f:
    f.write(code)

print(open(os.path.join(SRC_DIR, 'clean_text.py')).read())

import re

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'[^a-z0-9+#\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

